# Word2Vec

In [1]:
import os
from cbow.word2vec_cbow import Word2VecCBOW
from dataset.dataset import DatasetLoader

In [2]:
train_parquet_file_path = os.path.join("dataset", "train", "train.parquet")
train_dataset_loader = DatasetLoader(train_parquet_file_path)

train_sentences = train_dataset_loader.load_and_preprocess_data(train_parquet_file_path)

Rows in parquet: 36,718
Usable sentences: 21,076


## Build Training Pairs
We clip the dataset for faster experimentation, initialize the CBOW model, and generate context-target pairs.

In [3]:
max_sentences = 1000
embed_dim = 100
train_sentences_clipped = train_sentences[:max_sentences]

model = Word2VecCBOW(train_sentences_clipped, window_size=2, embed_dim=embed_dim)

contexts, target_words = model.get_cbow_examples()

print(f"Sentences used: {len(train_sentences_clipped):,}")
print(f"CBOW examples: {len(contexts):,}")
print("Example:")
print("context:", contexts[0])
print("target :", target_words[0])


Sentences used: 1,000
CBOW examples: 79,647
Example:
context: ['chronicles', 'iii']
target : valkyria


In [4]:
learning_rate = 0.1
epochs = 100

## Train The Model
Set optimization hyperparameters, then train with negative sampling for speed.

In [5]:
model.train(contexts, target_words, learning_rate=learning_rate, epochs=epochs, num_negative_samples=32, timeout=60*60)

Epoch 1/100 Epoch loss: 4.9309024281247265 Time: 19.88s
Epoch 2/100 Epoch loss: 2.9965939389531515 Time: 19.10s
Epoch 3/100 Epoch loss: 2.7579621376685575 Time: 18.80s
Epoch 4/100 Epoch loss: 2.5904702309379606 Time: 19.18s
Epoch 5/100 Epoch loss: 2.4221186266851973 Time: 18.62s
Epoch 6/100 Epoch loss: 2.265002065468382 Time: 19.42s
Epoch 7/100 Epoch loss: 2.1059603742878554 Time: 18.01s
Epoch 8/100 Epoch loss: 1.9591321386850855 Time: 17.74s
Epoch 9/100 Epoch loss: 1.8261180484636281 Time: 21.11s
Epoch 10/100 Epoch loss: 1.6957643000883271 Time: 21.84s
Epoch 11/100 Epoch loss: 1.5685372350077995 Time: 21.37s
Epoch 12/100 Epoch loss: 1.4482059127425522 Time: 20.91s
Epoch 13/100 Epoch loss: 1.3309515125358655 Time: 21.45s
Epoch 14/100 Epoch loss: 1.2261407316736577 Time: 20.84s
Epoch 15/100 Epoch loss: 1.1288146189821588 Time: 20.78s
Epoch 16/100 Epoch loss: 1.0375408820780876 Time: 17.72s
Epoch 17/100 Epoch loss: 0.9585136532575842 Time: 17.08s
Epoch 18/100 Epoch loss: 0.88492983048729

In [6]:
import joblib
import time

## Save And Reload
Persist the trained model to disk with a timestamped filename, then load it back for reuse.

In [7]:
now = time.strftime("%Y-%m-%d-%H-%M-%S")
joblib_filename = f"word2vec-{max_sentences}-max-sentences_{epochs}-epochs_{learning_rate}-learning-rate_{now}.joblib"
joblib_file = os.path.join("saved_models", joblib_filename)

print(f"Saving model to {joblib_filename} using joblib...")
joblib.dump(model, joblib_file)

Saving model to word2vec-1000-max-sentences_100-epochs_0.1-learning-rate_2026-03-16-19-03-43.joblib using joblib...


['saved_models/word2vec-1000-max-sentences_100-epochs_0.1-learning-rate_2026-03-16-19-03-43.joblib']

In [8]:
model_to_load = joblib_filename
path_to_model = os.path.join("saved_models", model_to_load)
model: Word2VecCBOW = joblib.load(path_to_model)
print(f"Model loaded from {model_to_load} successfully.")

Model loaded from word2vec-1000-max-sentences_100-epochs_0.1-learning-rate_2026-03-16-19-03-43.joblib successfully.


In [9]:
from dataset.dataset import DatasetLoader

In [10]:
test_dataset_file_path = os.path.join("dataset", "test", "test.parquet")

test_dataset_loader = DatasetLoader(test_dataset_file_path, column_name="text")

In [11]:
test_sentences = test_dataset_loader.load_and_preprocess_data(test_dataset_file_path)
print(f"Sentences in test set: {len(test_sentences):,}")
print(f"Example sentences: {test_sentences[:2]}")

Rows in parquet: 4,358
Usable sentences: 2,597
Sentences in test set: 2,597
Example sentences: ['robert boulter', "robert boulter is an english film television and theatre actor he had a guest starring role on the television series the bill in this was followed by a starring role in the play herons written by simon stephens which was performed in at the royal court theatre he had a guest role in the television series judge john deed in in boulter landed a role as craig in the episode teddy 's story of the television series the long firm he starred alongside actors mark strong and derek jacobi he was cast in the theatre productions of the philip ridley play mercury fur which was performed at the drum theatre in plymouth and the menier chocolate factory in london he was directed by john tiffany and starred alongside ben whishaw shane zaza harry kent fraser ayres sophie stanton and dominic hall"]


## Qualitative Evaluation
Run prediction examples on custom sentences and inspect top candidates with loss values.

In [12]:
sent = "the game 's main theme was how the characters regained their sense of self when stripped of their names and identities along with general themes focused on war and its consequences"
sent_words = sent.split()

for target_word_idx in [4, 7, 8, 9, 10, 15]:
    context, target_word = model.get_cbow_example(sent_words, target_word_idx=target_word_idx)

    predictions = model.predict(context)
    _, probs = model.forward(context)  # Unpack tuple: (h_hat, probs)
    loss = model.compute_loss(probs, target_word)

    print(f"Target word: '{target_word}'")
    print(f"Context words: {context}")
    print(f"Predictions: {predictions}")
    print(f"Loss: {loss:.6f}")
    print("-" * 40)


Target word: 'theme'
Context words: ["'s", 'main', 'was', 'how']
Predictions: [{'theme': 0.45717078286774904}, {'game': 0.27988176657589237}, {'the': 0.2295387780507083}]
Loss: 0.782698
----------------------------------------
Target word: 'the'
Context words: ['was', 'how', 'characters', 'regained']
Predictions: [{'the': 0.5258528247438036}, {'released': 0.46069992224131673}, {'he': 0.0035400166210457352}]
Loss: 0.642734
----------------------------------------
Target word: 'characters'
Context words: ['how', 'the', 'regained', 'their']
Predictions: [{'to': 0.44155383361653544}, {'at': 0.1736838374987204}, {'by': 0.08714388915205754}]
Loss: 3.379395
----------------------------------------
Target word: 'regained'
Context words: ['the', 'characters', 'their', 'sense']
Predictions: [{'of': 0.880530139831181}, {'in': 0.11034977824128413}, {'another': 0.0034373626787168646}]
Loss: 6.580865
----------------------------------------
Target word: 'their'
Context words: ['characters', 'regaine

In [13]:
sent2 = "i like the smell of the flowers"
sent2_words = sent2.split()

for target_word_idx in [2, 3]:
    context, target_word = model.get_cbow_example(sent2_words, target_word_idx=target_word_idx)

    predictions = model.predict(context)
    _, probs = model.forward(context)  # Unpack tuple: (h_hat, probs)
    loss = model.compute_loss(probs, target_word)

    print(f"Target word: '{target_word}'")
    print(f"Context words: {context}")
    print(f"Predictions: {predictions}")
    print(f"Loss: {loss:.6f}")
    print("-" * 40)

Target word: 'the'
Context words: ['i', 'like', 'smell', 'of']
Predictions: [{'the': 0.6898410741841535}, {'that': 0.1921352156811668}, {'heavier': 0.05240949380329957}]
Loss: 0.371294
----------------------------------------
Target word: 'smell'
Context words: ['like', 'the', 'of', 'the']
Predictions: [{'album': 0.9001966810494808}, {'depiction': 0.040532176927937626}, {'east': 0.0314378502680693}]
Loss: 16.373527
----------------------------------------


In [15]:
sent3 = "i like playing games with my friends"
sent3_words = sent2.split()

for target_word_idx in [3, 4]:
    context, target_word = model.get_cbow_example(sent2_words, target_word_idx=target_word_idx)

    predictions = model.predict(context)
    _, probs = model.forward(context)  # Unpack tuple: (h_hat, probs)
    loss = model.compute_loss(probs, target_word)

    print(f"Target word: '{target_word}'")
    print(f"Context words: {context}")
    print(f"Predictions: {predictions}")
    print(f"Loss: {loss:.6f}")
    print("-" * 40)

Target word: 'games'
Context words: ['like', 'playing', 'with', 'my']
Predictions: [{'songs': 0.29162239554184244}, {'metal': 0.1866147348045405}, {'i': 0.08814255912880418}]
Loss: 12.791956
----------------------------------------
Target word: 'with'
Context words: ['playing', 'games', 'my', 'friends']
Predictions: [{'or': 0.9993068864691329}, {'as': 0.00019978580029119422}, {'and': 0.00017713973530110403}]
Loss: 17.797190
----------------------------------------


## Considerations

The model is a simplified CBOW implementation and the training loop is basic. CBOW is a simple architecture and this embedding model was trained on a very small dataset, so the results are not outstanding. With more data and training, the embeddings would capture richer semantic relationships. Nevertheless, the code demonstrates the core mechanics of CBOW and gives reasonable predictions for the given context words.

Key points to improve:
- More training data and epochs for better embeddings.
- Adjust hyperparameters like learning rate and embedding dimension for better performance - I used considerably high learning rate and low embedding dimension for faster training and more visible results, but tuning these could yield better embeddings.